Install Dependencies

In [ ]:
!pip install transformers datasets seqeval evaluate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


Import Libraries

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


Download & Verify Files

In [ ]:
import urllib.request

# Direct raw file URLs from a working GitHub source
urls = {
    "train.txt": "https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.train",
    "valid.txt": "https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testa",
    "test.txt":  "https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testb",
}

for filename, url in urls.items():
    urllib.request.urlretrieve(url, filename)
    with open(filename, "r") as f:
        lines = f.readlines()
    print(f"✅ {filename}: {len(lines)} lines | First line: {lines[0].strip()!r}")

✅ train.txt: 219554 lines | First line: '-DOCSTART- -X- O O'
✅ valid.txt: 55045 lines | First line: '-DOCSTART- -X- O O'
✅ test.txt: 50351 lines | First line: '-DOCSTART- -X- -X- O'


Parse CoNLL Files

In [ ]:
def parse_conll_file(filepath):
    """
    CoNLL-2003 format per line: word  POS  chunk  NER
    Sentences separated by blank lines or -DOCSTART- lines.
    """
    sentences, pos_seqs, chunk_seqs = [], [], []
    words, pos_tags, chunk_tags = [], [], []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            # Skip document boundary markers
            if line.startswith("-DOCSTART-"):
                continue

            # Blank line = sentence boundary
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    pos_seqs.append(pos_tags)
                    chunk_seqs.append(chunk_tags)
                    words, pos_tags, chunk_tags = [], [], []
                continue

            parts = line.split()
            if len(parts) >= 3:
                words.append(parts[0])      # word
                pos_tags.append(parts[1])   # POS tag
                chunk_tags.append(parts[2]) # Chunk tag

    # Don't forget last sentence
    if words:
        sentences.append(words)
        pos_seqs.append(pos_tags)
        chunk_seqs.append(chunk_tags)

    return sentences, pos_seqs, chunk_seqs

# Parse
train_s, train_p, train_c = parse_conll_file("train.txt")
valid_s, valid_p, valid_c = parse_conll_file("valid.txt")
test_s,  test_p,  test_c  = parse_conll_file("test.txt")

print(f"Train : {len(train_s)} sentences")
print(f"Valid : {len(valid_s)} sentences")
print(f"Test  : {len(test_s)} sentences")

# Sanity check
print(f"\nSample tokens     : {train_s[0]}")
print(f"Sample POS tags   : {train_p[0]}")
print(f"Sample Chunk tags : {train_c[0]}")

Train : 14041 sentences
Valid : 3250 sentences
Test  : 3453 sentences

Sample tokens     : ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Sample POS tags   : ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.']
Sample Chunk tags : ['I-NP', 'I-VP', 'I-NP', 'I-NP', 'I-VP', 'I-VP', 'I-NP', 'I-NP', 'O']


Build HuggingFace DatasetDict

In [ ]:
from datasets import Dataset, DatasetDict

# Build label vocabularies from training data only
all_pos_labels   = sorted(set(t for seq in train_p for t in seq))
all_chunk_labels = sorted(set(t for seq in train_c for t in seq))

pos_label2id   = {l: i for i, l in enumerate(all_pos_labels)}
chunk_label2id = {l: i for i, l in enumerate(all_chunk_labels)}

def encode(seqs, label2id):
    """Convert string label sequences to integer id sequences."""
    return [[label2id[t] for t in seq] for seq in seqs]

def make_split(sents, pos, chunk):
    return Dataset.from_dict({
        "tokens":     sents,
        "pos_tags":   encode(pos,   pos_label2id),
        "chunk_tags": encode(chunk, chunk_label2id),
    })

dataset = DatasetDict({
    "train":      make_split(train_s, train_p, train_c),
    "validation": make_split(valid_s, valid_p, valid_c),
    "test":       make_split(test_s,  test_p,  test_c),
})

print("✅ Dataset built successfully!")
print(dataset)

✅ Dataset built successfully!
DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 3453
    })
})


View Label Lists

In [ ]:
pos_labels   = all_pos_labels
chunk_labels = all_chunk_labels

print(f"POS Labels   ({len(pos_labels)})  : {pos_labels}")
print(f"Chunk Labels ({len(chunk_labels)}): {chunk_labels}")

POS Labels   (45)  : ['"', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Chunk Labels (17): ['B-ADJP', 'B-ADVP', 'B-NP', 'B-PP', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']


Task 2: Preprocessing – Choose Task (POS or Chunking)

In [ ]:
# ============================================================
# CHOOSE TASK: Set task = "pos" for POS Tagging
#              Set task = "chunk" for Chunking
# ============================================================
TASK = "chunk"   # Change to "pos" if needed

if TASK == "pos":
    label_list = all_pos_labels    # Already built in Cell 4B
    label_key  = "pos_tags"
else:
    label_list = all_chunk_labels  # Already built in Cell 4B
    label_key  = "chunk_tags"

print(f"Task   : {TASK.upper()}")
print(f"Labels ({len(label_list)}): {label_list}")

Task   : CHUNK
Labels (17): ['B-ADJP', 'B-ADVP', 'B-NP', 'B-PP', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']


 Tokenization & Label Alignment

In [ ]:
# Load BERT tokenizer (fast tokenizer required for word_ids())
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    """
    Tokenize inputs and align labels with subword tokens.
    Subword tokens beyond the first get label -100 (ignored in loss).
    Special tokens ([CLS], [SEP]) also get -100.
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True  # Input is already word-tokenized
    )

    all_labels = []
    for i, labels in enumerate(examples[label_key]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens → ignore
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First token of a word → assign real label
                label_ids.append(labels[word_idx])
            else:
                # Subsequent subword tokens → ignore
                label_ids.append(-100)
            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

# Apply preprocessing to all splits
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)
print("✅ Tokenization & label alignment done!")
print("Sample keys:", tokenized_dataset["train"][0].keys())

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

✅ Tokenization & label alignment done!
Sample keys: dict_keys(['tokens', 'pos_tags', 'chunk_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'])


Task 3: Model Setup

In [ ]:
# Create label <-> id mappings
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# Load DistilBERT for token classification
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"Number of labels: {len(label_list)}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded: distilbert-base-uncased
Number of labels: 17


Task 4: Training Setup

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [label_list[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

training_args = TrainingArguments(
    output_dir=f"./bert-{TASK}-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,        # ✅ replaces deprecated 'tokenizer'
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("✅ Trainer configured. Ready to train!")

✅ Trainer configured. Ready to train!


Train the Model

In [ ]:
# Start training
trainer.train()
print("✅ Training complete!")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.155695,0.157681,0.912190,0.896064,0.904055,0.960301
2,0.126924,0.139560,0.917677,0.909119,0.913378,0.963981
3,0.096667,0.137027,0.920622,0.910285,0.915424,0.964877


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


✅ Training complete!


Task 5: Evaluation

In [ ]:
# Evaluate on the test set
results = trainer.evaluate(tokenized_dataset["test"])

print("\n📊 Evaluation Results on Test Set:")
print(f"  Precision : {results['eval_precision']:.4f}")
print(f"  Recall    : {results['eval_recall']:.4f}")
print(f"  F1 Score  : {results['eval_f1']:.4f}")
print(f"  Accuracy  : {results['eval_accuracy']:.4f}")

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



📊 Evaluation Results on Test Set:
  Precision : 0.9122
  Recall    : 0.8978
  F1 Score  : 0.9049
  Accuracy  : 0.9622


Save the Model

In [ ]:
# Save model and tokenizer locally (for reuse or download)
model.save_pretrained(f"./bert-{TASK}-final")
tokenizer.save_pretrained(f"./bert-{TASK}-final")
print(f"✅ Model saved to ./bert-{TASK}-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ./bert-chunk-final


Task 6: Inference on Custom Sentences

In [ ]:
from transformers import pipeline

# Load inference pipeline from saved model
nlp_pipeline = pipeline(
    "token-classification",
    model=f"./bert-{TASK}-final",
    tokenizer=f"./bert-{TASK}-final",
    aggregation_strategy="simple"   # Merge subword tokens
)

# Test sentences
test_sentences = [
    "John works at Google in California.",
    "The quick brown fox jumps over the lazy dog.",
    "Apple is looking at buying a startup in the UK."
]

print(f"🔍 Running {TASK.upper()} inference:\n")
for sentence in test_sentences:
    print(f"Input : {sentence}")
    output = nlp_pipeline(sentence)
    for entity in output:
        print(f"  Word: {entity['word']:<20} Tag: {entity['entity_group']:<12} Score: {entity['score']:.3f}")
    print()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

🔍 Running CHUNK inference:

Input : John works at Google in California.
  Word: john                 Tag: NP           Score: 0.998
  Word: works                Tag: VP           Score: 0.634
  Word: at                   Tag: PP           Score: 0.990
  Word: google               Tag: NP           Score: 0.996
  Word: in                   Tag: PP           Score: 0.997
  Word: california           Tag: NP           Score: 0.996

Input : The quick brown fox jumps over the lazy dog.
  Word: the quick brown fox  Tag: NP           Score: 0.985
  Word: jumps                Tag: VP           Score: 0.917
  Word: over                 Tag: PP           Score: 0.824
  Word: the lazy dog         Tag: NP           Score: 0.993

Input : Apple is looking at buying a startup in the UK.
  Word: apple                Tag: NP           Score: 0.998
  Word: is looking           Tag: VP           Score: 0.995
  Word: at                   Tag: PP           Score: 0.960
  Word: buying               Tag: VP 

Task 7: Comparison – POS Tagging vs Chunking

In [ ]:
comparison = {
    "Aspect": [
        "Goal",
        "Granularity",
        "Output Example",
        "Complexity",
        "Use Case"
    ],
    "POS Tagging": [
        "Assign grammatical role to each word",
        "Word-level (token-level)",
        "John/NNP works/VBZ at/IN",
        "Easier — direct word classification",
        "Grammar analysis, spell check, parsing"
    ],
    "Chunking": [
        "Group words into meaningful phrases",
        "Phrase-level (span-level)",
        "[NP John] [VP works] [PP at Google]",
        "Harder — requires phrase boundary detection",
        "Information extraction, NER, QA systems"
    ]
}

import pandas as pd
df = pd.DataFrame(comparison)
print(df.to_string(index=False))

        Aspect                            POS Tagging                                    Chunking
          Goal   Assign grammatical role to each word         Group words into meaningful phrases
   Granularity               Word-level (token-level)                   Phrase-level (span-level)
Output Example               John/NNP works/VBZ at/IN         [NP John] [VP works] [PP at Google]
    Complexity    Easier — direct word classification Harder — requires phrase boundary detection
      Use Case Grammar analysis, spell check, parsing     Information extraction, NER, QA systems


Task 8: Report / Blog Summary

In [ ]:
report = """
╔══════════════════════════════════════════════════════════════════╗
║        REPORT: POS Tagging vs Chunking with DistilBERT          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  DIFFERENCES:                                                    ║
║  • POS Tagging labels individual tokens (NN, VBZ, IN ...)       ║
║  • Chunking groups tokens into named phrases (NP, VP, PP ...)   ║
║                                                                  ║
║  CHALLENGES FACED:                                               ║
║  • Subword tokenization — BERT splits words like "working"       ║
║    into "work" + "##ing"; only the first token gets a label      ║
║  • Special tokens [CLS]/[SEP] must be masked with -100           ║
║  • Label alignment is crucial for correct training               ║
║                                                                  ║
║  OBSERVATIONS:                                                   ║
║  • DistilBERT is 40% smaller than BERT but performs similarly   ║
║  • seqeval ignores -100 labels automatically during evaluation  ║
║  • Chunking F1 is generally slightly lower than POS tagging     ║
║    because phrase boundaries are more ambiguous                  ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(report)


╔══════════════════════════════════════════════════════════════════╗
║        REPORT: POS Tagging vs Chunking with DistilBERT          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  DIFFERENCES:                                                    ║
║  • POS Tagging labels individual tokens (NN, VBZ, IN ...)       ║
║  • Chunking groups tokens into named phrases (NP, VP, PP ...)   ║
║                                                                  ║
║  CHALLENGES FACED:                                               ║
║  • Subword tokenization — BERT splits words like "working"       ║
║    into "work" + "##ing"; only the first token gets a label      ║
║  • Special tokens [CLS]/[SEP] must be masked with -100           ║
║  • Label alignment is crucial for correct training               ║
║                                                                  ║
║  OBSERVATIONS:                    